In [2]:
import torch
import pandas as pd
import numpy as np

import lightning.pytorch as pl
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting import TemporalFusionTransformer

print("PyTorch:", torch.__version__)
print("Lightning:", pl.__version__)

PyTorch: 2.8.0
Lightning: 2.6.0


/Users/riyaagarwal/Library/Python/3.9/lib/python/site-packages/pytorch_forecasting/models/base/_base_model.py:28: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
import pandas as pd

df = pd.read_csv("../data/train.csv")

df.head()

/var/folders/xd/fgpxnmwx71n9tz4rn52jp45h0000gn/T/ipykernel_41320/1523523810.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/train.csv")


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [6]:
df.shape

(1017209, 9)

In [7]:
df.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday'],
      dtype='object')

PRE-PROCESSING

In [12]:
df = df.sort_values(["Store", "Date"]).reset_index(drop=True)

In [13]:
df[["Store", "Date", "Sales"]].head(20)

,Store,Date,Sales
0,1,2013-01-02,5530
1,1,2013-01-03,4327
2,1,2013-01-04,4486
3,1,2013-01-05,4997
4,1,2013-01-07,7176
5,1,2013-01-08,5580
6,1,2013-01-09,5471
7,1,2013-01-10,4892
8,1,2013-01-11,4881
9,1,2013-01-12,4952


In [14]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.isnull().sum())

(844392, 9)
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']
Store                     int64
DayOfWeek                 int64
Date             datetime64[ns]
Sales                     int64
Customers                 int64
Open                      int64
Promo                     int64
StateHoliday             object
SchoolHoliday             int64
dtype: object
Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64


In [15]:
print("Number of stores:", df["Store"].nunique())

print("Date range:")
print(df["Date"].min(), "to", df["Date"].max())

print("\nOpen values:")
print(df["Open"].value_counts())

print("\nPromo values:")
print(df["Promo"].value_counts())

print("\nStateHoliday values:")
print(df["StateHoliday"].value_counts())

print("\nDayOfWeek values:")
print(df["DayOfWeek"].value_counts().sort_index())

Number of stores: 1115
Date range:
2013-01-01 00:00:00 to 2015-07-31 00:00:00

Open values:
Open
1    844392
Name: count, dtype: int64

Promo values:
Promo
0    467496
1    376896
Name: count, dtype: int64

StateHoliday values:
StateHoliday
0    731342
0    112140
a       694
b       145
c        71
Name: count, dtype: int64

DayOfWeek values:
DayOfWeek
1    137560
2    143961
3    141936
4    134644
5    138640
6    144058
7      3593
Name: count, dtype: int64


In [16]:
# Make a copy so we don't accidentally modify the original dataframe
data = df.copy()

# Make sure Date is datetime
data["Date"] = pd.to_datetime(data["Date"])

# Clean StateHoliday
data["StateHoliday"] = data["StateHoliday"].astype(str)

# Convert categorical features to string
data["Store"] = data["Store"].astype(str)
data["DayOfWeek"] = data["DayOfWeek"].astype(str)
data["Promo"] = data["Promo"].astype(str)
data["SchoolHoliday"] = data["SchoolHoliday"].astype(str)

# Sort by store and date
data = data.sort_values(["Store", "Date"]).reset_index(drop=True)

# Create continuous time index
data["time_idx"] = (
    data["Date"] - data["Date"].min()
).dt.days

# Check
print(data.head())
print(data.dtypes)

  Store DayOfWeek       Date  Sales  Customers  Open Promo StateHoliday  \
0     1         3 2013-01-02   5530        668     1     0            0   
1     1         4 2013-01-03   4327        578     1     0            0   
2     1         5 2013-01-04   4486        619     1     0            0   
3     1         6 2013-01-05   4997        635     1     0            0   
4     1         1 2013-01-07   7176        785     1     1            0   

  SchoolHoliday  time_idx  
0             1         1  
1             1         2  
2             1         3  
3             1         4  
4             1         6  
Store                    object
DayOfWeek                object
Date             datetime64[ns]
Sales                     int64
Customers                 int64
Open                      int64
Promo                    object
StateHoliday             object
SchoolHoliday            object
time_idx                  int64
dtype: object


In [17]:
print(data[["Date", "time_idx"]].head(20))

         Date  time_idx
0  2013-01-02         1
1  2013-01-03         2
2  2013-01-04         3
3  2013-01-05         4
4  2013-01-07         6
5  2013-01-08         7
6  2013-01-09         8
7  2013-01-10         9
8  2013-01-11        10
9  2013-01-12        11
10 2013-01-14        13
11 2013-01-15        14
12 2013-01-16        15
13 2013-01-17        16
14 2013-01-18        17
15 2013-01-19        18
16 2013-01-21        20
17 2013-01-22        21
18 2013-01-23        22
19 2013-01-24        23


## 6. Feature Selection for M4 — Temporal Fusion Transformer

For the initial M4 implementation, the following features from the Rossmann
Store Sales dataset are used:

- **Store:** identifies each individual store/time series.
- **DayOfWeek:** categorical temporal feature.
- **Promo:** indicates whether a promotion is active.
- **StateHoliday:** indicates the type of state holiday.
- **SchoolHoliday:** indicates whether a school holiday is active.
- **Customers:** historical customer count, treated as a time-varying feature.
- **Sales:** target variable to be forecast.
- **time_idx:** continuous integer representation of the calendar timeline.

The target variable is **Sales**. Store is used as the time-series group identifier.
The temporal and categorical variables are provided to the TFT model as covariates.

## 7. Train-Validation Split

A chronological train-validation split is used to prevent look-ahead data leakage.
The final 60 days of the dataset are reserved for validation, while earlier
observations are used for training.

In [18]:
max_date = data["Date"].max()

validation_start = max_date - pd.Timedelta(days=60)

train_data = data[data["Date"] < validation_start].copy()
val_data = data[data["Date"] >= validation_start].copy()

print("Training:")
print(train_data["Date"].min(), "to", train_data["Date"].max())
print("Rows:", len(train_data))

print("\nValidation:")
print(val_data["Date"].min(), "to", val_data["Date"].max())
print("Rows:", len(val_data))

Training:
2013-01-01 00:00:00 to 2015-05-31 00:00:00
Rows: 785781

Validation:
2015-06-01 00:00:00 to 2015-07-31 00:00:00
Rows: 58611


In [19]:
print("Training stores:", train_data["Store"].nunique())
print("Validation stores:", val_data["Store"].nunique())

print("\nTraining max date:", train_data["Date"].max())
print("Validation min date:", val_data["Date"].min())

Training stores: 1115
Validation stores: 1115

Training max date: 2015-05-31 00:00:00
Validation min date: 2015-06-01 00:00:00


## 8. Creating the Time Series Dataset for M4

The Temporal Fusion Transformer requires the data to be organized as
time-series sequences. Each store is treated as a separate time series,
with `Store` used as the group identifier.

The model uses a 30-day historical encoder window to predict sales for
the following 7 days. The input variables are divided into static/group
identifiers, known time-varying covariates, and unknown time-varying
variables.

- **Group identifier:** Store
- **Target:** Sales
- **Known time-varying covariates:** DayOfWeek, Promo, StateHoliday,
  SchoolHoliday, and time_idx
- **Unknown time-varying variables:** Sales and Customers
- **Encoder length:** 30 days
- **Prediction length:** 7 days

In [20]:
from pytorch_forecasting import TimeSeriesDataSet

In [21]:
max_encoder_length = 30
max_prediction_length = 7

In [23]:
training = TimeSeriesDataSet(
    train_data,

    # Time index
    time_idx="time_idx",

    # Target
    target="Sales",

    # Each store is treated as a separate time series
    group_ids=["Store"],

    # Historical input window
    min_encoder_length=max_encoder_length,
    max_encoder_length=max_encoder_length,

    # Forecast horizon
    min_prediction_length=max_prediction_length,
    max_prediction_length=max_prediction_length,

    # Allow gaps in calendar dates
    allow_missing_timesteps=True,

    # Static categorical feature
    static_categoricals=[
        "Store"
    ],

    # Known time-varying categorical features
    time_varying_known_categoricals=[
        "DayOfWeek",
        "Promo",
        "StateHoliday",
        "SchoolHoliday"
    ],

    # Known time-varying numerical feature
    time_varying_known_reals=[
        "time_idx"
    ],

    # Unknown time-varying variables
    time_varying_unknown_reals=[
        "Sales",
        "Customers"
    ],
)

In [24]:
print(training)
print("Number of training samples:", len(training))

TimeSeriesDataSet[length=603342](
	time_idx='time_idx',
	target='Sales',
	group_ids=['Store'],
	weight=None,
	max_encoder_length=30,
	min_encoder_length=30,
	min_prediction_idx=np.int64(0),
	min_prediction_length=7,
	max_prediction_length=7,
	static_categoricals=['Store'],
	static_reals=None,
	time_varying_known_categoricals=['DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday'],
	time_varying_known_reals=['time_idx'],
	time_varying_unknown_categoricals=None,
	time_varying_unknown_reals=['Sales', 'Customers'],
	variable_groups=None,
	constant_fill_strategy=None,
	allow_missing_timesteps=True,
	lags=None,
	add_relative_time_idx=False,
	add_target_scales=False,
	add_encoder_length=False,
	target_normalizer=NaNLabelEncoder(add_nan=False, warn=True),
	categorical_encoders={'__group_id__Store': NaNLabelEncoder(add_nan=False, warn=True), 'Store': NaNLabelEncoder(add_nan=False, warn=True), 'DayOfWeek': NaNLabelEncoder(add_nan=False, warn=True), 'Promo': NaNLabelEncoder(add_nan=False, warn=Tr

### 8.4 Checking Categorical Feature Consistency

Before creating the validation `TimeSeriesDataSet`, the categorical
features are checked to ensure that the validation set does not contain
categories that were absent from the training set.

In [28]:
categorical_cols = [
    "Store",
    "DayOfWeek",
    "Promo",
    "StateHoliday",
    "SchoolHoliday"
]

for col in categorical_cols:
    train_values = set(train_data[col].unique())
    val_values = set(val_data[col].unique())

    unseen = val_values - train_values

    print(f"\n{col}")
    print("Training unique values:", len(train_values))
    print("Validation unique values:", len(val_values))
    print("Unseen validation values:", unseen)


Store
Training unique values: 1115
Validation unique values: 1115
Unseen validation values: set()

DayOfWeek
Training unique values: 7
Validation unique values: 7
Unseen validation values: set()

Promo
Training unique values: 2
Validation unique values: 2
Unseen validation values: set()

StateHoliday
Training unique values: 4
Validation unique values: 2
Unseen validation values: set()

SchoolHoliday
Training unique values: 2
Validation unique values: 2
Unseen validation values: set()


In [29]:
print("17961 occurrences in validation:")

for col in categorical_cols:
    matches = val_data[val_data[col].astype(str) == "17961"]

    if len(matches) > 0:
        print(col, len(matches))
        print(matches[[col, "Store", "Date"]].head())

17961 occurrences in validation:


In [30]:
print("Categorical encoders:")
print(training.categorical_encoders)

Categorical encoders:
None


In [32]:
print("Time varying known categoricals:")
print(training.time_varying_known_categoricals)

print("\nTime varying known reals:")
print(training.time_varying_known_reals)

print("\nTime varying unknown reals:")
print(training.time_varying_unknown_reals)

print("\nStatic categoricals:")
print(training.static_categoricals)

print("\nGroup IDs:")
print(training.group_ids)

print("\nTarget:")
print(training.target)

Time varying known categoricals:
['DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday']

Time varying known reals:
['time_idx']

Time varying unknown reals:
['Sales', 'Customers']

Static categoricals:
['Store']

Group IDs:
['Store']

Target:
Sales


In [33]:
params = training.get_parameters()

print("\nCategorical encoders in parameters:")
print(params.get("categorical_encoders"))


Categorical encoders in parameters:
{'__group_id__Store': NaNLabelEncoder(add_nan=False, warn=True), 'Store': NaNLabelEncoder(add_nan=False, warn=True), 'DayOfWeek': NaNLabelEncoder(add_nan=False, warn=True), 'Promo': NaNLabelEncoder(add_nan=False, warn=True), 'StateHoliday': NaNLabelEncoder(add_nan=False, warn=True), 'SchoolHoliday': NaNLabelEncoder(add_nan=False, warn=True)}


In [34]:
print("17961 in validation time_idx:",
      17961 in val_data["time_idx"].values)

print("17961 in training time_idx:",
      17961 in train_data["time_idx"].values)

17961 in validation time_idx: False
17961 in training time_idx: False


### 8.4 Validation Dataset

The validation dataset is created using the same configuration as the
training dataset. This ensures that categorical encodings and other
dataset settings remain consistent between training and validation.

The validation data contains the final 60 days of the chronological
dataset and is not used during model training.

In [36]:
print("Target:", training.target)
print("Target normalizer:", training.target_normalizer)
print("Sales dtype:", train_data["Sales"].dtype)
print("Sales sample:")
print(train_data["Sales"].head())

Target: Sales
Target normalizer: NaNLabelEncoder(add_nan=False, warn=True)
Sales dtype: int64
Sales sample:
0    5530
1    4327
2    4486
3    4997
4    7176
Name: Sales, dtype: int64


In [37]:
print("17961 as Sales:")
print((train_data["Sales"] == 17961).sum())
print((validation_data["Sales"] == 17961).sum())

17961 as Sales:
0
1


In [38]:
from pytorch_forecasting.data import TorchNormalizer

In [39]:
training = TimeSeriesDataSet(
    train_data,
    time_idx="time_idx",
    target="Sales",
    group_ids=["Store"],

    min_encoder_length=max_encoder_length,
    max_encoder_length=max_encoder_length,
    min_prediction_length=max_prediction_length,
    max_prediction_length=max_prediction_length,

    allow_missing_timesteps=True,

    static_categoricals=["Store"],

    time_varying_known_categoricals=[
        "DayOfWeek",
        "Promo",
        "StateHoliday",
        "SchoolHoliday"
    ],

    time_varying_known_reals=[
        "time_idx"
    ],

    time_varying_unknown_reals=[
        "Sales",
        "Customers"
    ],

    target_normalizer=TorchNormalizer()
)

print("Training dataset recreated successfully!")
print("Target normalizer:", training.target_normalizer)

Training dataset recreated successfully!
Target normalizer: TorchNormalizer(method='standard', center=True, transformation=None, method_kwargs=None)


In [40]:
validation = TimeSeriesDataSet.from_dataset(
    training,
    validation_data,
    predict=True,
    stop_randomization=True
)

print("Validation dataset created successfully!")
print("Number of validation samples:", len(validation))

Validation dataset created successfully!
Number of validation samples: 1114


/Users/riyaagarwal/Library/Python/3.9/lib/python/site-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1847: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 1 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__Store': '183'}]
  warnings.warn(


Step 9 - create dataloaders

In [41]:
# Step 9: Create DataLoaders

batch_size = 64

train_loader = training.to_dataloader(
    train=True,
    batch_size=batch_size,
    num_workers=0
)

val_loader = validation.to_dataloader(
    train=False,
    batch_size=batch_size,
    num_workers=0
)

print("Train DataLoader created successfully!")
print("Validation DataLoader created successfully!")

Train DataLoader created successfully!
Validation DataLoader created successfully!


In [43]:
print("Target type:", type(batch[1]))
print("Number of target elements:", len(batch[1]))

print("Target shape:", batch[1][0].shape)

Target type: <class 'tuple'>
Number of target elements: 2
Target shape: torch.Size([64, 7])


step 10 - build tft model

In [49]:
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.001,
    hidden_size=16,
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=8,
    loss=QuantileLoss(),
    log_interval=-1,
    reduce_on_plateau_patience=3,
)

print(tft)

TemporalFusionTransformer(
  	"attention_head_size":               2
  	"categorical_groups":                {}
  	"causal_attention":                  True
  	"dataset_parameters":                {'time_idx': 'time_idx', 'target': 'Sales', 'group_ids': ['Store'], 'weight': None, 'max_encoder_length': 30, 'min_encoder_length': 30, 'min_prediction_idx': np.int64(0), 'min_prediction_length': 7, 'max_prediction_length': 7, 'static_categoricals': ['Store'], 'static_reals': None, 'time_varying_known_categoricals': ['DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday'], 'time_varying_known_reals': ['time_idx'], 'time_varying_unknown_categoricals': None, 'time_varying_unknown_reals': ['Sales', 'Customers'], 'variable_groups': None, 'constant_fill_strategy': None, 'allow_missing_timesteps': True, 'lags': None, 'add_relative_time_idx': False, 'add_target_scales': False, 'add_encoder_length': False, 'target_normalizer': TorchNormalizer(method='standard', center=True, transformation=None, method

step - 11

In [50]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min"
)

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="cpu",
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback],
    enable_model_summary=True
)

print("CPU Trainer created successfully!")

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores


CPU Trainer created successfully!


In [51]:
trainer.fit(
    tft,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader
)


   | Name                               | Type                            | Params | Mode  | FLOPs
--------------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train | 0    
1  | logging_metrics                    | ModuleList                      | 0      | train | 0    
2  | input_embeddings                   | MultiEmbedding                  | 17.9 K | train | 0    
3  | prescalers                         | ModuleDict                      | 48     | train | 0    
4  | static_variable_selection          | VariableSelectionNetwork        | 48     | train | 0    
5  | encoder_variable_selection         | VariableSelectionNetwork        | 2.3 K  | train | 0    
6  | decoder_variable_selection         | VariableSelectionNetwork        | 1.0 K  | train | 0    
7  | static_context_variable_selection  | GatedResidualNetwork            | 1.1 K  | train | 0    
8  

Epoch 3: 100%|██████████| 9427/9427 [11:40<00:00, 13.46it/s, v_num=2, train_loss_step=342.0, val_loss=479.0, train_loss_epoch=324.0]


In [52]:
trainer.callback_metrics

{'train_loss': tensor(324.3530),
 'train_loss_step': tensor(341.6812),
 'val_loss': tensor(479.3595),
 'val_SMAPE': tensor(0.1116),
 'val_MAE': tensor(831.4567),
 'val_RMSE': tensor(1208.4437),
 'val_MAPE': tensor(0.1109),
 'train_loss_epoch': tensor(324.3530)}

In [53]:
trainer.logged_metrics

{'train_loss_step': tensor(341.6812),
 'val_loss': tensor(479.3595),
 'val_SMAPE': tensor(0.1116),
 'val_MAE': tensor(831.4567),
 'val_RMSE': tensor(1208.4437),
 'val_MAPE': tensor(0.1109),
 'train_loss_epoch': tensor(324.3530)}

In [54]:
trainer.callback_metrics

{'train_loss': tensor(324.3530),
 'train_loss_step': tensor(341.6812),
 'val_loss': tensor(479.3595),
 'val_SMAPE': tensor(0.1116),
 'val_MAE': tensor(831.4567),
 'val_RMSE': tensor(1208.4437),
 'val_MAPE': tensor(0.1109),
 'train_loss_epoch': tensor(324.3530)}

In [55]:
trainer.logger.log_dir

'/Users/riyaagarwal/Desktop/M4_TFT/notebooks/lightning_logs/version_2'

In [56]:
import pandas as pd

metrics = pd.read_csv(
    trainer.logger.log_dir + "/metrics.csv"
)

metrics

,epoch,step,train_loss_epoch,train_loss_step,val_MAE,val_MAPE,val_RMSE,val_SMAPE,val_loss
0,0,49,NaN,1045.978760,NaN,NaN,NaN,NaN,NaN
1,0,99,NaN,895.728821,NaN,NaN,NaN,NaN,NaN
2,0,149,NaN,849.426758,NaN,NaN,NaN,NaN,NaN
3,0,199,NaN,1029.627686,NaN,NaN,NaN,NaN,NaN
4,0,249,NaN,883.082153,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
757,3,37599,NaN,376.313049,NaN,NaN,NaN,NaN,NaN
758,3,37649,NaN,349.358673,NaN,NaN,NaN,NaN,NaN
759,3,37699,NaN,320.440765,NaN,NaN,NaN,NaN,NaN
760,3,37707,NaN,NaN,831.456726,0.110944,1208.443726,0.111587,479.359467


In [57]:
metrics[["epoch", "train_loss_epoch", "val_loss", "val_SMAPE", "val_MAE", "val_RMSE", "val_MAPE"]]

,epoch,train_loss_epoch,val_loss,val_SMAPE,val_MAE,val_RMSE,val_MAPE
0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,0,NaN,NaN,NaN,NaN,NaN,NaN
2,0,NaN,NaN,NaN,NaN,NaN,NaN
3,0,NaN,NaN,NaN,NaN,NaN,NaN
4,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
757,3,NaN,NaN,NaN,NaN,NaN,NaN
758,3,NaN,NaN,NaN,NaN,NaN,NaN
759,3,NaN,NaN,NaN,NaN,NaN,NaN
760,3,NaN,479.359467,0.111587,831.456726,1208.443726,0.110944


In [58]:
metrics[
    metrics["val_loss"].notna()
][["epoch", "val_loss", "val_SMAPE", "val_MAE", "val_RMSE", "val_MAPE"]]

,epoch,val_loss,val_SMAPE,val_MAE,val_RMSE,val_MAPE
188,0,441.153900,0.109945,794.668335,1107.857788,0.113947
379,1,467.655029,0.109626,808.985718,1176.549316,0.110069
569,2,466.503662,0.112180,825.422485,1157.612061,0.114789
760,3,479.359467,0.111587,831.456726,1208.443726,0.110944


In [59]:
print("Best checkpoint:", trainer.checkpoint_callback.best_model_path)
print("Best score:", trainer.checkpoint_callback.best_model_score)

Best checkpoint: /Users/riyaagarwal/Desktop/M4_TFT/notebooks/lightning_logs/version_2/checkpoints/epoch=3-step=37708.ckpt
Best score: None


prediction

In [60]:
predictions = tft.predict(
    val_loader,
    mode="prediction"
)

print(predictions.shape)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/riyaagarwal/Library/Python/3.9/lib/python/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


TypeError: Cannot convert a MPS Tensor to float64 dtype as the MPS framework doesn't support float64. Please use float32 instead.

In [61]:
predictions = tft.predict(
    val_loader,
    mode="prediction",
    trainer_kwargs={
        "accelerator": "cpu",
        "devices": 1
    }
)

print(predictions.shape)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/riyaagarwal/Library/Python/3.9/lib/python/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


torch.Size([1114, 7])


In [62]:
actuals = torch.cat([
    y[0] if isinstance(y, tuple) else y
    for x, y in val_loader
])

print(actuals.shape)

torch.Size([1114, 7])


In [63]:
import numpy as np

y_true = actuals.cpu().numpy().reshape(-1)
y_pred = predictions.cpu().numpy().reshape(-1)

mae = np.mean(np.abs(y_true - y_pred))
rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

# RMSPE — exclude zero-sales observations
mask = y_true != 0
rmspe = np.sqrt(
    np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2)
)

print(f"MAE:   {mae:.2f}")
print(f"RMSE:  {rmse:.2f}")
print(f"RMSPE: {rmspe:.4f} ({rmspe*100:.2f}%)")

MAE:   831.46
RMSE:  1212.27
RMSPE: 0.1605 (16.05%)
